In [1]:
%pip install --upgrade pip
%pip install  duckdb
%pip install   numpy
%pip install  pandas
%pip install fastavro
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 38.3 MB/s eta 0:00:00 0:00:01
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 30.9 MB/s eta 0:00:00 0:00:01
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import time

TABLE_WRITE_TIME="TABLE_WRITE_TIME_SEC"
TABLE_FULL_SCAN_TIME="TABLE_FULL_SCAN_TIME_SEC"
prev_time_start = {}

time_metrics = {}

def measureTimeStart(metric):
    prev_time_start[metric] = time.perf_counter()

def measureTimeEnd(metric, additional_tag):
    time_metrics[additional_tag + "_" + metric] = time.perf_counter() - prev_time_start[metric]


In [107]:
 spark.stop()

In [108]:
from pyspark.sql import SparkSession
from pyspark import SparkConf, SparkContext
# from pyspark.sql.functions import col, count, avg, sum, min, max, desc
import sys
import time
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MinIO Write") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.spark:spark-avro_2.12:3.5.3,org.apache.spark:spark-core_2.12:3.5.3,org.apache.spark:spark-sql_2.12:3.5.3") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio-server:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.socket.timeout", "60000") \
    .config("spark.hadoop.fs.s3.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3.socket.timeout", "60000") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.memoryOverhead", "2g") \
    .config("spark.driver.memoryOverhead", "1g") \
    .getOrCreate()


In [ ]:
# START TABLE CREATION

In [ ]:
# records = table.to_dict(DF_DICT_TYPE)
# measureTimeStart(TABLE_WRITE_TIME)
# parsed_schema = parse_schema(schema)

# file_name=get_file_name(cur_schema, base_table_name,compression)
# table_name = get_table_name(cur_schema, base_table_name,compression)
# #with open(file_name, 'wb') as file:
# #    writer(file, parsed_schema, records)
# spark_df = spark.createDataFrame(table)
# spark_df.write.format("avro").mode("overwrite").option("compression", compression).save(file_name) # get_s3_path(file_name))

# measureTimeEnd(TABLE_WRITE_TIME, time_metrics[cur_schema][table_name])


In [4]:
%pip install pyarrow matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 MB 45.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 29.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 63.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [110]:
%%time

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/workspace/data.csv")

df.count()

CPU times: user 40.2 ms, sys: 0 ns, total: 40.2 ms
Wall time: 10.2 s


7160831

In [111]:
df.show(5)

+-------------+-------------+--------+---------+---------+-----+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------+---------+---------+-----------+----+-----+---+------+----+---+---+-----------+---------+----+---+----+----+---+---+-------+----+-------+--------+---------+--------+------------+------+----------+----------+----------+----------+------+--------+
|Header_Length|Protocol Type|Duration|     Rate|    Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|ack_count|syn_count|fin_count|  rst_count|HTTP|HTTPS|DNS|Telnet|SMTP|SSH|IRC|        TCP|      UDP|DHCP|ARP|ICMP|IGMP|IPv|LLC|Tot sum| Min|    Max|     AVG|      Std|Tot size|         IAT|Number|  Magnitue|    Radius|Covariance|  Variance|Weight|   label|
+-------------+-------------+--------+---------+---------+-----+---------------+---------------+---------------+---------------+--------

In [112]:
rdd = df.rdd #self.sc.parallelize(df, num_partitions=200)

In [8]:
%%time

rdd \
    .map(lambda row: (row['Protocol Type'], (row['Header_Length'], 1))) \
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1])) \
    .mapValues(lambda x: x[0] / x[1]) \
    .collect()

CPU times: user 25.7 ms, sys: 11.9 ms, total: 37.6 ms
Wall time: 22.6 s


[(16.125, 37921.09534177214),
 (13.8125, 162930.6526618705),
 (1.1601562, 1253.2081801168754),
 (0.9902344, 0.006689308919078559),
 (1.0615234, 34.851852),
 (6.4414062, 40305.83196206284),
 (5.9414062, 122212.01804136054),
 (5.7617188, 186995.6868016194),
 (16.734375, 24716.40316504854),
 (12.6015625, 724408.4194675327),
 (5.7382812, 1100046.9342857143),
 (5.75, 859348.3891836734),
 (5.8515625, 485654.8565306122),
 (15.5234375, 25359.215517241384),
 (6.1484375, 1124765.4536666668),
 (15.9296875, 23112.490825531913),
 (15.7265625, 42778.432222222225),
 (5.6484375, 1442145.7441666669),
 (5.1289062, 824937.9528571429),
 (5.5585938, 625917.719),
 (6.2617188, 718288.4544444444),
 (6.0585938, 171125.33642857143),
 (6.2382812, 59967.785057142864),
 (5.140625, 1486852.1379999998),
 (15.1171875, 76187.8625),
 (4.6289062, 6761.895),
 (7.359375, 5750.39),
 (6.859375, 11401.8475),
 (6.25, 3039.921609090909),
 (7.9804688, 38740.549402985074),
 (10.1796875, 481070.3521428571),
 (11.390625, 264681.07

In [10]:
from pyspark.sql.functions import avg

In [11]:
%%time

df.groupBy("Protocol Type") \
    .agg(avg("Header_Length").alias("mean_header_length")).collect()


CPU times: user 15.1 ms, sys: 4.04 ms, total: 19.1 ms
Wall time: 4.99 s


[Row(Protocol Type=0.97998047, mean_header_length=0.0),
 Row(Protocol Type=2.359375, mean_header_length=65087.96206896552),
 Row(Protocol Type=4.359375, mean_header_length=124817.08836734694),
 Row(Protocol Type=2.1191406, mean_header_length=13750.356933534747),
 Row(Protocol Type=2.109375, mean_header_length=44526.412),
 Row(Protocol Type=3.3496094, mean_header_length=333093.51249999995),
 Row(Protocol Type=1.390625, mean_header_length=5977.0244),
 Row(Protocol Type=2.3300781, mean_header_length=12310.785999999998),
 Row(Protocol Type=6.1445312, mean_header_length=405.14284999999995),
 Row(Protocol Type=2.2695312, mean_header_length=20847.71857142857),
 Row(Protocol Type=8.75, mean_header_length=220704.3212121212),
 Row(Protocol Type=11.0625, mean_header_length=391688.62148148153),
 Row(Protocol Type=6.078125, mean_header_length=151850.84166666667),
 Row(Protocol Type=5.9609375, mean_header_length=261078.52983606557),
 Row(Protocol Type=2.3203125, mean_header_length=18675.719333333334

In [12]:
%%time

rdd.sortBy(lambda row: row["Rate"]).count()


CPU times: user 71.5 ms, sys: 41.1 ms, total: 113 ms
Wall time: 1min 56s


7160831

In [13]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, col

In [15]:
%%time

window_spec = Window.rowsBetween(-10, Window.currentRow)
df.repartition(20).withColumn("moveing_avg_10", avg("Rate").over(window_spec)).show(100)

26/03/06 11:16:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:16:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:16:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:16:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:16:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:16:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 1

+-------------+-------------+--------+----------+----------+-----+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------+----------+-----------+-----------+----+-----------+---+------+----+---+---+-----------+---------+----+-----------+----------+----+----------+----------+-------+--------+--------+--------+-----------+--------+-----------+------+----------+-----------+----------+-----------+------+---------+------------------+
|Header_Length|Protocol Type|Duration|      Rate|     Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|ack_count| syn_count|  fin_count|  rst_count|HTTP|      HTTPS|DNS|Telnet|SMTP|SSH|IRC|        TCP|      UDP|DHCP|        ARP|      ICMP|IGMP|       IPv|       LLC|Tot sum|     Min|     Max|     AVG|        Std|Tot size|        IAT|Number|  Magnitue|     Radius|Covariance|   Variance|Weight|    label|    moveing_avg_10|
+-

In [16]:
%%time

window_spec = Window.rowsBetween(-10, Window.currentRow)
df.repartition(5).withColumn("moveing_avg_10", avg("Rate").over(window_spec)).show(100)

26/03/06 11:17:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 1

+-------------+-------------+--------+----------+----------+-----+---------------+---------------+---------------+---------------+---------------+---------------+---------------+-----------+----------+-----------+-----------+----+----------+---+------+----+---+---+----------+----------+----+-----------+-----------+----+---------+---------+-------+--------+--------+--------+-----------+--------+-----------+------+----------+-----------+-----------+-----------+------+-----------------+------------------+
|Header_Length|Protocol Type|Duration|      Rate|     Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|  ack_count| syn_count|  fin_count|  rst_count|HTTP|     HTTPS|DNS|Telnet|SMTP|SSH|IRC|       TCP|       UDP|DHCP|        ARP|       ICMP|IGMP|      IPv|      LLC|Tot sum|     Min|     Max|     AVG|        Std|Tot size|        IAT|Number|  Magnitue|     Radius| Covariance|   Variance|Weight|            label|    

In [17]:
%%time

window_spec = Window.rowsBetween(-10, Window.currentRow)
df.repartition(100).withColumn("moveing_avg_10", avg("Rate").over(window_spec)).show(100)

26/03/06 11:17:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 1

+-------------+-------------+--------+---------+---------+-----+---------------+---------------+---------------+---------------+---------------+---------------+---------------+----------+----------+-----------+-----------+----+-----------+---+------+----+---+---+-----------+----------+----+---+-----------+----+---+---+-------+--------+--------+--------+-----------+--------+-----------+------+----------+-----------+-----------+-----------+------+-----------------+------------------+
|Header_Length|Protocol Type|Duration|     Rate|    Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number| ack_count| syn_count|  fin_count|  rst_count|HTTP|      HTTPS|DNS|Telnet|SMTP|SSH|IRC|        TCP|       UDP|DHCP|ARP|       ICMP|IGMP|IPv|LLC|Tot sum|     Min|     Max|     AVG|        Std|Tot size|        IAT|Number|  Magnitue|     Radius| Covariance|   Variance|Weight|            label|    moveing_avg_10|
+-------------+-----------

In [18]:
%%time

window_spec = Window.rowsBetween(-10, Window.currentRow)
df.repartition(1000).withColumn("moveing_avg_10", avg("Rate").over(window_spec)).show(100)

26/03/06 11:17:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:17:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:18:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 1

+-------------+-------------+--------+----------+----------+-----+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------+---------+-----------+----------+----+-----------+---+------+----+---+---+-----------+---------+----+---+----+----+---+---+-------+--------+--------+--------+-----------+--------+-----------+------+----------+-----------+----------+-----------+------+---------+------------------+
|Header_Length|Protocol Type|Duration|      Rate|     Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|ack_count|syn_count|  fin_count| rst_count|HTTP|      HTTPS|DNS|Telnet|SMTP|SSH|IRC|        TCP|      UDP|DHCP|ARP|ICMP|IGMP|IPv|LLC|Tot sum|     Min|     Max|     AVG|        Std|Tot size|        IAT|Number|  Magnitue|     Radius|Covariance|   Variance|Weight|    label|    moveing_avg_10|
+-------------+-------------+--------+----------+----------+--

In [24]:
%%time

window_spec = Window.orderBy("Header_Length").rowsBetween(-10, Window.currentRow)
res = df.repartition(50).withColumn("moveing_avg_10", avg("Rate").over(window_spec))
res.explain(True)
res.show(100)

26/03/06 11:22:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:22:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:22:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:22:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:22:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:22:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


== Parsed Logical Plan ==
'Project [Header_Length#17, Protocol Type#18, Duration#19, Rate#20, Srate#21, Drate#22, fin_flag_number#23, syn_flag_number#24, rst_flag_number#25, psh_flag_number#26, ack_flag_number#27, ece_flag_number#28, cwr_flag_number#29, ack_count#30, syn_count#31, fin_count#32, rst_count#33, HTTP#34, HTTPS#35, DNS#36, Telnet#37, SMTP#38, SSH#39, IRC#40, ... 23 more fields]
+- Repartition 50, true
   +- Relation [Header_Length#17,Protocol Type#18,Duration#19,Rate#20,Srate#21,Drate#22,fin_flag_number#23,syn_flag_number#24,rst_flag_number#25,psh_flag_number#26,ack_flag_number#27,ece_flag_number#28,cwr_flag_number#29,ack_count#30,syn_count#31,fin_count#32,rst_count#33,HTTP#34,HTTPS#35,DNS#36,Telnet#37,SMTP#38,SSH#39,IRC#40,... 22 more fields] csv

== Analyzed Logical Plan ==
Header_Length: double, Protocol Type: double, Duration: double, Rate: double, Srate: double, Drate: int, fin_flag_number: double, syn_flag_number: double, rst_flag_number: double, psh_flag_number: doub

26/03/06 11:22:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:22:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:22:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:22:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------+-------------+--------+---------+---------+-----+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------+---------+---------+---------+----+-----+---+------+----+---+---+---+---+----+---+----+----+---+---+-------+----+----+----+---+--------+-----------+------+---------+------+----------+--------+------+---------+------------------+
|Header_Length|Protocol Type|Duration|     Rate|    Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|ack_count|syn_count|fin_count|rst_count|HTTP|HTTPS|DNS|Telnet|SMTP|SSH|IRC|TCP|UDP|DHCP|ARP|ICMP|IGMP|IPv|LLC|Tot sum| Min| Max| AVG|Std|Tot size|        IAT|Number| Magnitue|Radius|Covariance|Variance|Weight|    label|    moveing_avg_10|
+-------------+-------------+--------+---------+---------+-----+---------------+---------------+---------------+---------------+---------------+---------------+----------

# Test dataframe window avg vs sql window avg

итоговый вывод - идентичны

In [37]:
%%time

window_spec = Window.rowsBetween(-10, Window.currentRow)
res = df.repartition(50).withColumn("moveing_avg_10", avg("Rate").over(window_spec))
res.explain(True)
res.show(100)

26/03/06 11:43:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:43:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:43:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:43:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:43:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:43:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


== Parsed Logical Plan ==
'Project [Header_Length#17, Protocol Type#18, Duration#19, Rate#20, Srate#21, Drate#22, fin_flag_number#23, syn_flag_number#24, rst_flag_number#25, psh_flag_number#26, ack_flag_number#27, ece_flag_number#28, cwr_flag_number#29, ack_count#30, syn_count#31, fin_count#32, rst_count#33, HTTP#34, HTTPS#35, DNS#36, Telnet#37, SMTP#38, SSH#39, IRC#40, ... 23 more fields]
+- Repartition 50, true
   +- Relation [Header_Length#17,Protocol Type#18,Duration#19,Rate#20,Srate#21,Drate#22,fin_flag_number#23,syn_flag_number#24,rst_flag_number#25,psh_flag_number#26,ack_flag_number#27,ece_flag_number#28,cwr_flag_number#29,ack_count#30,syn_count#31,fin_count#32,rst_count#33,HTTP#34,HTTPS#35,DNS#36,Telnet#37,SMTP#38,SSH#39,IRC#40,... 22 more fields] csv

== Analyzed Logical Plan ==
Header_Length: double, Protocol Type: double, Duration: double, Rate: double, Srate: double, Drate: int, fin_flag_number: double, syn_flag_number: double, rst_flag_number: double, psh_flag_number: doub

26/03/06 11:43:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:43:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:43:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:43:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------+-------------+--------+----------+----------+-----+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------+----------+-----------+-----------+----+-----------+---+------+----+---+---+-----------+-----------+----+---+----------+----+---+---+-------+--------+--------+--------+-----------+--------+-----------+------+----------+-----------+----------+-----------+------+---------+------------------+
|Header_Length|Protocol Type|Duration|      Rate|     Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|ack_count| syn_count|  fin_count|  rst_count|HTTP|      HTTPS|DNS|Telnet|SMTP|SSH|IRC|        TCP|        UDP|DHCP|ARP|      ICMP|IGMP|IPv|LLC|Tot sum|     Min|     Max|     AVG|        Std|Tot size|        IAT|Number|  Magnitue|     Radius|Covariance|   Variance|Weight|    label|    moveing_avg_10|
+-------------+-------------+--------+----

# у pyspark иногда возникали пробелмы с тем, что вроде бы с 137 ошибкой падал один из executor и итогово получалось + 10 сек к результату
# но как только пошёл в ui смотреть - это перестало воспроизводиться - магия

In [35]:
%%time

df.createOrReplaceTempView("df_view")

# Use Spark SQL with window function
result = spark.sql("""
    SELECT 
        *,
        AVG(Rate) OVER (
            ROWS BETWEEN 10 PRECEDING AND CURRENT ROW
        ) as moving_avg_10
    FROM df_view
""")

result.explain(True)
result.show(100)

26/03/06 11:42:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:42:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:42:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:42:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:42:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:42:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


== Parsed Logical Plan ==
'Project [*, 'AVG('Rate) windowspecdefinition(specifiedwindowframe(RowFrame, -10, currentrow$())) AS moving_avg_10#6178]
+- 'UnresolvedRelation [df_view], [], false

== Analyzed Logical Plan ==
Header_Length: double, Protocol Type: double, Duration: double, Rate: double, Srate: double, Drate: int, fin_flag_number: double, syn_flag_number: double, rst_flag_number: double, psh_flag_number: double, ack_flag_number: double, ece_flag_number: double, cwr_flag_number: double, ack_count: double, syn_count: double, fin_count: double, rst_count: double, HTTP: double, HTTPS: double, DNS: double, Telnet: double, SMTP: double, SSH: double, IRC: double, ... 23 more fields
Project [Header_Length#17, Protocol Type#18, Duration#19, Rate#20, Srate#21, Drate#22, fin_flag_number#23, syn_flag_number#24, rst_flag_number#25, psh_flag_number#26, ack_flag_number#27, ece_flag_number#28, cwr_flag_number#29, ack_count#30, syn_count#31, fin_count#32, rst_count#33, HTTP#34, HTTPS#35, DNS#3

26/03/06 11:42:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:42:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------+-------------+--------+---------+---------+-----+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------+----------+-----------+-----------+----+-----------+---+------+----+---+---+---+---+----+---+----+----+---+---+-------+--------+--------+--------+-----------+--------+-----------+---------+----------+-----------+-----------+-----------+--------+--------+------------------+
|Header_Length|Protocol Type|Duration|     Rate|    Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|ack_count| syn_count|  fin_count|  rst_count|HTTP|      HTTPS|DNS|Telnet|SMTP|SSH|IRC|TCP|UDP|DHCP|ARP|ICMP|IGMP|IPv|LLC|Tot sum|     Min|     Max|     AVG|        Std|Tot size|        IAT|   Number|  Magnitue|     Radius| Covariance|   Variance|  Weight|   label|     moving_avg_10|
+-------------+-------------+--------+---------+---------+-----+---------------+

In [34]:
%%time

df.repartition(50).createOrReplaceTempView("df_view")

# Use Spark SQL with window function
result = spark.sql("""
    SELECT 
        *,
        AVG(Rate) OVER (
            ROWS BETWEEN 10 PRECEDING AND CURRENT ROW
        ) as moving_avg_10
    FROM df_view
""")

result.explain(True)
result.show(100)

26/03/06 11:41:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:41:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:41:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:41:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:41:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:41:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


== Parsed Logical Plan ==
'Project [*, 'AVG('Rate) windowspecdefinition(specifiedwindowframe(RowFrame, -10, currentrow$())) AS moving_avg_10#5894]
+- 'UnresolvedRelation [df_view], [], false

== Analyzed Logical Plan ==
Header_Length: double, Protocol Type: double, Duration: double, Rate: double, Srate: double, Drate: int, fin_flag_number: double, syn_flag_number: double, rst_flag_number: double, psh_flag_number: double, ack_flag_number: double, ece_flag_number: double, cwr_flag_number: double, ack_count: double, syn_count: double, fin_count: double, rst_count: double, HTTP: double, HTTPS: double, DNS: double, Telnet: double, SMTP: double, SSH: double, IRC: double, ... 23 more fields
Project [Header_Length#17, Protocol Type#18, Duration#19, Rate#20, Srate#21, Drate#22, fin_flag_number#23, syn_flag_number#24, rst_flag_number#25, psh_flag_number#26, ack_flag_number#27, ece_flag_number#28, cwr_flag_number#29, ack_count#30, syn_count#31, fin_count#32, rst_count#33, HTTP#34, HTTPS#35, DNS#3

26/03/06 11:42:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:42:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:42:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/06 11:42:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------+-------------+--------+----------+----------+-----+---------------+---------------+---------------+---------------+---------------+---------------+---------------+----------+----------+-----------+-----------+-----------+-----------+---+------+----+---+---+----------+----------+----+---+-----------+----+---+---+-------+--------+--------+--------+-----------+--------+-----------+------+----------+-----------+-----------+-----------+------+-----------------+------------------+
|Header_Length|Protocol Type|Duration|      Rate|     Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number| ack_count| syn_count|  fin_count|  rst_count|       HTTP|      HTTPS|DNS|Telnet|SMTP|SSH|IRC|       TCP|       UDP|DHCP|ARP|       ICMP|IGMP|IPv|LLC|Tot sum|     Min|     Max|     AVG|        Std|Tot size|        IAT|Number|  Magnitue|     Radius| Covariance|   Variance|Weight|            label|     moving_avg_10|
+---------

# Првоерка на то что filter действительно переставляются

итог:
фильтры по изачальным колонкам переставились в начало
фильтры по аггрегированным колонкам 

в RDD не применяется перемещение фильтров даже внутри одного stage (посмотрел в ui)

In [54]:
from pyspark.sql.functions import sum, avg, count, when, col, min, max, stddev, collect_list, first, last, approx_count_distinct
from pyspark.sql.functions import sum, avg, count, when, col, min, max, stddev, mean, approx_count_distinct, collect_list, countDistinct
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType, StringType
from pyspark.sql import Row
import numpy as np

In [74]:
%%time

# Сначала агрегируем
aggregated_df = df.groupBy("Protocol Type", "label").agg(
    # Базовые метрики Rate
    sum("Rate").alias("total_rate"),
)

# Потом применяем фильтры
filtered_result = aggregated_df.filter(
    (col("total_rate") > 1000000) &
    (col("Protocol Type").isin([6, 17]))
).orderBy(col("total_rate").desc())

filtered_result.explain(True)

# filtered_result.show(1000)

filtered_result.count()

== Parsed Logical Plan ==
'Sort ['total_rate DESC NULLS LAST], true
+- Filter ((total_rate#17294 > cast(1000000 as double)) AND cast(Protocol Type#9958 as double) IN (cast(6 as double),cast(17 as double)))
   +- Aggregate [Protocol Type#9958, label#10002], [Protocol Type#9958, label#10002, sum(Rate#9960) AS total_rate#17294]
      +- Relation [Header_Length#9957,Protocol Type#9958,Duration#9959,Rate#9960,Srate#9961,Drate#9962,fin_flag_number#9963,syn_flag_number#9964,rst_flag_number#9965,psh_flag_number#9966,ack_flag_number#9967,ece_flag_number#9968,cwr_flag_number#9969,ack_count#9970,syn_count#9971,fin_count#9972,rst_count#9973,HTTP#9974,HTTPS#9975,DNS#9976,Telnet#9977,SMTP#9978,SSH#9979,IRC#9980,... 22 more fields] csv

== Analyzed Logical Plan ==
Protocol Type: double, label: string, total_rate: double
Sort [total_rate#17294 DESC NULLS LAST], true
+- Filter ((total_rate#17294 > cast(1000000 as double)) AND cast(Protocol Type#9958 as double) IN (cast(6 as double),cast(17 as double)))

CPU times: user 5.42 ms, sys: 8.12 ms, total: 13.5 ms
Wall time: 2.48 s


15

In [ ]:
%%time

result = df.rdd \
    .map(lambda row: ((row["Protocol Type"], row["label"]), row["Rate"])) \
    .reduceByKey(lambda a, b: a + b) \
    .filter(lambda x: x[1] > 1000000 and x[0][0] in [6, 17])

result.count()
print(result.toDebugString().decode())

(13) PythonRDD[448] at RDD at PythonRDD.scala:53 []
 |   MapPartitionsRDD[446] at mapPartitions at PythonRDD.scala:160 []
 |   ShuffledRDD[445] at partitionBy at <unknown>:0 []
 +-(13) PairwiseRDD[444] at reduceByKey at <timed exec>:1 []
    |   PythonRDD[443] at reduceByKey at <timed exec>:1 []
    |   MapPartitionsRDD[322] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |   MapPartitionsRDD[321] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |   SQLExecutionRDD[320] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |   MapPartitionsRDD[319] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |   FileScanRDD[318] at javaToPython at NativeMethodAccessorImpl.java:0 []
CPU times: user 30.6 ms, sys: 25 ms, total: 55.6 ms
Wall time: 18.3 s


In [89]:
%%time

result = df.rdd \
    .map(lambda row: ((row["Protocol Type"], row["label"]), row["Rate"])) \
    .reduceByKey(lambda a, b: a + b) \
    .filter(lambda x: x[0][0] == 6) \
    .repartition(10) \
    .reduceByKey(lambda a, b: a + b) \
    .filter(lambda x: x[1] > 1000000)

result.count()
print(result.toDebugString().decode())

(10) PythonRDD[486] at RDD at PythonRDD.scala:53 []
 |   MapPartitionsRDD[484] at mapPartitions at PythonRDD.scala:160 []
 |   ShuffledRDD[483] at partitionBy at <unknown>:0 []
 +-(10) PairwiseRDD[482] at reduceByKey at <timed exec>:1 []
    |   PythonRDD[481] at reduceByKey at <timed exec>:1 []
    |   MapPartitionsRDD[480] at coalesce at NativeMethodAccessorImpl.java:0 []
    |   CoalescedRDD[479] at coalesce at NativeMethodAccessorImpl.java:0 []
    |   ShuffledRDD[478] at coalesce at NativeMethodAccessorImpl.java:0 []
    +-(13) MapPartitionsRDD[477] at coalesce at NativeMethodAccessorImpl.java:0 []
       |   PythonRDD[476] at RDD at PythonRDD.scala:53 []
       |   MapPartitionsRDD[475] at mapPartitions at PythonRDD.scala:160 []
       |   ShuffledRDD[474] at partitionBy at <unknown>:0 []
       +-(13) PairwiseRDD[473] at reduceByKey at <timed exec>:1 []
          |   PythonRDD[472] at reduceByKey at <timed exec>:1 []
          |   MapPartitionsRDD[322] at javaToPython at NativeM

In [91]:
%%time

result = df.rdd \
    .map(lambda row: ((row["Protocol Type"], row["label"]), row["Rate"])) \
    .reduceByKey(lambda a, b: a + b) \
    .repartition(10) \
    .reduceByKey(lambda a, b: a + b) \
    .filter(lambda x: x[1] > 1000000 and x[0][0] == 6)

result.count()
print(result.toDebugString().decode())

(10) PythonRDD[501] at RDD at PythonRDD.scala:53 []
 |   MapPartitionsRDD[499] at mapPartitions at PythonRDD.scala:160 []
 |   ShuffledRDD[498] at partitionBy at <unknown>:0 []
 +-(10) PairwiseRDD[497] at reduceByKey at <timed exec>:1 []
    |   PythonRDD[496] at reduceByKey at <timed exec>:1 []
    |   MapPartitionsRDD[495] at coalesce at NativeMethodAccessorImpl.java:0 []
    |   CoalescedRDD[494] at coalesce at NativeMethodAccessorImpl.java:0 []
    |   ShuffledRDD[493] at coalesce at NativeMethodAccessorImpl.java:0 []
    +-(13) MapPartitionsRDD[492] at coalesce at NativeMethodAccessorImpl.java:0 []
       |   PythonRDD[491] at RDD at PythonRDD.scala:53 []
       |   MapPartitionsRDD[490] at mapPartitions at PythonRDD.scala:160 []
       |   ShuffledRDD[489] at partitionBy at <unknown>:0 []
       +-(13) PairwiseRDD[488] at reduceByKey at <timed exec>:1 []
          |   PythonRDD[487] at reduceByKey at <timed exec>:1 []
          |   MapPartitionsRDD[322] at javaToPython at NativeM

In [ ]:
%%time

result = df.rdd \
    .map(lambda row: ((row["Protocol Type"], row["label"]), row["Rate"])) \
    .reduceByKey(lambda a, b: a + b) \
    .reduceByKey(lambda a, b: a + b) \
    .filter(lambda x: x[1] > 1000000) \
    .filter(lambda x:  False) \
    .repartition(10)
# не перемещает даже внутри stage

result.count()
print(result.toDebugString().decode())

(10) MapPartitionsRDD[540] at coalesce at NativeMethodAccessorImpl.java:0 []
 |   CoalescedRDD[539] at coalesce at NativeMethodAccessorImpl.java:0 []
 |   ShuffledRDD[538] at coalesce at NativeMethodAccessorImpl.java:0 []
 +-(13) MapPartitionsRDD[537] at coalesce at NativeMethodAccessorImpl.java:0 []
    |   PythonRDD[536] at RDD at PythonRDD.scala:53 []
    |   MapPartitionsRDD[535] at mapPartitions at PythonRDD.scala:160 []
    |   ShuffledRDD[534] at partitionBy at <unknown>:0 []
    +-(13) PairwiseRDD[533] at reduceByKey at <timed exec>:1 []
       |   PythonRDD[532] at reduceByKey at <timed exec>:1 []
       |   MapPartitionsRDD[322] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |   MapPartitionsRDD[321] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |   SQLExecutionRDD[320] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |   MapPartitionsRDD[319] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |   FileScanRDD[318] at javaToPy

In [115]:
%%timeit -r 5

joined = df.alias("df1").join(df.alias("df2"), "Rate") \
           .filter(col("df2.Rate") > 1000) \
           .filter(col("df1.Rate") < 100000) \
           .agg(sum("df1.Rate").alias("total"))

joined.show(1000)
joined.explain(True)

+--------------------+
|               total|
+--------------------+
|2.238876031062219E14|
+--------------------+

== Parsed Logical Plan ==
'Aggregate [sum('df1.Rate) AS total#19786]
+- Filter (Rate#18926 < cast(100000 as double))
   +- Project [Rate#18926, Header_Length#18923, Protocol Type#18924, Duration#18925, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#18935, ack_count#18936, syn_count#18937, fin_count#18938, rst_count#18939, HTTP#18940, HTTPS#18941, DNS#18942, Telnet#18943, SMTP#18944, SSH#18945, IRC#18946, ... 67 more fields]
      +- Filter (Rate#19650 > cast(1000 as double))
         +- Project [Rate#18926, Header_Length#18923, Protocol Type#18924, Duration#18925, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#

+--------------------+
|               total|
+--------------------+
|2.238876031062219...|
+--------------------+

== Parsed Logical Plan ==
'Aggregate [sum('df1.Rate) AS total#20029]
+- Filter (Rate#18926 < cast(100000 as double))
   +- Project [Rate#18926, Header_Length#18923, Protocol Type#18924, Duration#18925, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#18935, ack_count#18936, syn_count#18937, fin_count#18938, rst_count#18939, HTTP#18940, HTTPS#18941, DNS#18942, Telnet#18943, SMTP#18944, SSH#18945, IRC#18946, ... 67 more fields]
      +- Filter (Rate#19893 > cast(1000 as double))
         +- Project [Rate#18926, Header_Length#18923, Protocol Type#18924, Duration#18925, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#

+--------------------+
|               total|
+--------------------+
|2.238876031062219...|
+--------------------+

== Parsed Logical Plan ==
'Aggregate [sum('df1.Rate) AS total#20272]
+- Filter (Rate#18926 < cast(100000 as double))
   +- Project [Rate#18926, Header_Length#18923, Protocol Type#18924, Duration#18925, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#18935, ack_count#18936, syn_count#18937, fin_count#18938, rst_count#18939, HTTP#18940, HTTPS#18941, DNS#18942, Telnet#18943, SMTP#18944, SSH#18945, IRC#18946, ... 67 more fields]
      +- Filter (Rate#20136 > cast(1000 as double))
         +- Project [Rate#18926, Header_Length#18923, Protocol Type#18924, Duration#18925, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#

+--------------------+
|               total|
+--------------------+
|2.238876031062219...|
+--------------------+

== Parsed Logical Plan ==
'Aggregate [sum('df1.Rate) AS total#20515]
+- Filter (Rate#18926 < cast(100000 as double))
   +- Project [Rate#18926, Header_Length#18923, Protocol Type#18924, Duration#18925, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#18935, ack_count#18936, syn_count#18937, fin_count#18938, rst_count#18939, HTTP#18940, HTTPS#18941, DNS#18942, Telnet#18943, SMTP#18944, SSH#18945, IRC#18946, ... 67 more fields]
      +- Filter (Rate#20379 > cast(1000 as double))
         +- Project [Rate#18926, Header_Length#18923, Protocol Type#18924, Duration#18925, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#

+--------------------+
|               total|
+--------------------+
|2.238876031062219...|
+--------------------+

== Parsed Logical Plan ==
'Aggregate [sum('df1.Rate) AS total#20758]
+- Filter (Rate#18926 < cast(100000 as double))
   +- Project [Rate#18926, Header_Length#18923, Protocol Type#18924, Duration#18925, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#18935, ack_count#18936, syn_count#18937, fin_count#18938, rst_count#18939, HTTP#18940, HTTPS#18941, DNS#18942, Telnet#18943, SMTP#18944, SSH#18945, IRC#18946, ... 67 more fields]
      +- Filter (Rate#20622 > cast(1000 as double))
         +- Project [Rate#18926, Header_Length#18923, Protocol Type#18924, Duration#18925, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#

+--------------------+
|               total|
+--------------------+
|2.238876031062219...|
+--------------------+

== Parsed Logical Plan ==
'Aggregate [sum('df1.Rate) AS total#21001]
+- Filter (Rate#18926 < cast(100000 as double))
   +- Project [Rate#18926, Header_Length#18923, Protocol Type#18924, Duration#18925, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#18935, ack_count#18936, syn_count#18937, fin_count#18938, rst_count#18939, HTTP#18940, HTTPS#18941, DNS#18942, Telnet#18943, SMTP#18944, SSH#18945, IRC#18946, ... 67 more fields]
      +- Filter (Rate#20865 > cast(1000 as double))
         +- Project [Rate#18926, Header_Length#18923, Protocol Type#18924, Duration#18925, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#

In [116]:
%%timeit -r 5

df.createOrReplaceTempView("df_table")

sql_result = spark.sql("""
    SELECT SUM(df1.Rate) as total
    FROM df_table df1
    JOIN df_table df2 ON df1.Rate = df2.Rate
    WHERE df2.Rate > 1000 
      AND df1.Rate < 100000
""")

sql_result.explain(True)

sql_result.show(1000)

== Parsed Logical Plan ==
'Project ['SUM('df1.Rate) AS total#21105]
+- 'Filter (('df2.Rate > 1000) AND ('df1.Rate < 100000))
   +- 'Join Inner, ('df1.Rate = 'df2.Rate)
      :- 'SubqueryAlias df1
      :  +- 'UnresolvedRelation [df_table], [], false
      +- 'SubqueryAlias df2
         +- 'UnresolvedRelation [df_table], [], false

== Analyzed Logical Plan ==
total: double
Aggregate [sum(Rate#18926) AS total#21105]
+- Filter ((Rate#21109 > cast(1000 as double)) AND (Rate#18926 < cast(100000 as double)))
   +- Join Inner, (Rate#18926 = Rate#21109)
      :- SubqueryAlias df1
      :  +- SubqueryAlias df_table
      :     +- View (`df_table`, [Header_Length#18923,Protocol Type#18924,Duration#18925,Rate#18926,Srate#18927,Drate#18928,fin_flag_number#18929,syn_flag_number#18930,rst_flag_number#18931,psh_flag_number#18932,ack_flag_number#18933,ece_flag_number#18934,cwr_flag_number#18935,ack_count#18936,syn_count#18937,fin_count#18938,rst_count#18939,HTTP#18940,HTTPS#18941,DNS#18942,Telnet#1894

+--------------------+
|               total|
+--------------------+
|2.238876031062219E14|
+--------------------+

== Parsed Logical Plan ==
'Project ['SUM('df1.Rate) AS total#21165]
+- 'Filter (('df2.Rate > 1000) AND ('df1.Rate < 100000))
   +- 'Join Inner, ('df1.Rate = 'df2.Rate)
      :- 'SubqueryAlias df1
      :  +- 'UnresolvedRelation [df_table], [], false
      +- 'SubqueryAlias df2
         +- 'UnresolvedRelation [df_table], [], false

== Analyzed Logical Plan ==
total: double
Aggregate [sum(Rate#18926) AS total#21165]
+- Filter ((Rate#21169 > cast(1000 as double)) AND (Rate#18926 < cast(100000 as double)))
   +- Join Inner, (Rate#18926 = Rate#21169)
      :- SubqueryAlias df1
      :  +- SubqueryAlias df_table
      :     +- View (`df_table`, [Header_Length#18923,Protocol Type#18924,Duration#18925,Rate#18926,Srate#18927,Drate#18928,fin_flag_number#18929,syn_flag_number#18930,rst_flag_number#18931,psh_flag_number#18932,ack_flag_number#18933,ece_flag_number#18934,cwr_flag_numbe

+--------------------+
|               total|
+--------------------+
|2.238876031062219...|
+--------------------+

== Parsed Logical Plan ==
'Project ['SUM('df1.Rate) AS total#21225]
+- 'Filter (('df2.Rate > 1000) AND ('df1.Rate < 100000))
   +- 'Join Inner, ('df1.Rate = 'df2.Rate)
      :- 'SubqueryAlias df1
      :  +- 'UnresolvedRelation [df_table], [], false
      +- 'SubqueryAlias df2
         +- 'UnresolvedRelation [df_table], [], false

== Analyzed Logical Plan ==
total: double
Aggregate [sum(Rate#18926) AS total#21225]
+- Filter ((Rate#21229 > cast(1000 as double)) AND (Rate#18926 < cast(100000 as double)))
   +- Join Inner, (Rate#18926 = Rate#21229)
      :- SubqueryAlias df1
      :  +- SubqueryAlias df_table
      :     +- View (`df_table`, [Header_Length#18923,Protocol Type#18924,Duration#18925,Rate#18926,Srate#18927,Drate#18928,fin_flag_number#18929,syn_flag_number#18930,rst_flag_number#18931,psh_flag_number#18932,ack_flag_number#18933,ece_flag_number#18934,cwr_flag_numbe

+--------------------+
|               total|
+--------------------+
|2.238876031062219...|
+--------------------+

== Parsed Logical Plan ==
'Project ['SUM('df1.Rate) AS total#21285]
+- 'Filter (('df2.Rate > 1000) AND ('df1.Rate < 100000))
   +- 'Join Inner, ('df1.Rate = 'df2.Rate)
      :- 'SubqueryAlias df1
      :  +- 'UnresolvedRelation [df_table], [], false
      +- 'SubqueryAlias df2
         +- 'UnresolvedRelation [df_table], [], false

== Analyzed Logical Plan ==
total: double
Aggregate [sum(Rate#18926) AS total#21285]
+- Filter ((Rate#21289 > cast(1000 as double)) AND (Rate#18926 < cast(100000 as double)))
   +- Join Inner, (Rate#18926 = Rate#21289)
      :- SubqueryAlias df1
      :  +- SubqueryAlias df_table
      :     +- View (`df_table`, [Header_Length#18923,Protocol Type#18924,Duration#18925,Rate#18926,Srate#18927,Drate#18928,fin_flag_number#18929,syn_flag_number#18930,rst_flag_number#18931,psh_flag_number#18932,ack_flag_number#18933,ece_flag_number#18934,cwr_flag_numbe

+--------------------+
|               total|
+--------------------+
|2.238876031062219E14|
+--------------------+

== Parsed Logical Plan ==
'Project ['SUM('df1.Rate) AS total#21345]
+- 'Filter (('df2.Rate > 1000) AND ('df1.Rate < 100000))
   +- 'Join Inner, ('df1.Rate = 'df2.Rate)
      :- 'SubqueryAlias df1
      :  +- 'UnresolvedRelation [df_table], [], false
      +- 'SubqueryAlias df2
         +- 'UnresolvedRelation [df_table], [], false

== Analyzed Logical Plan ==
total: double
Aggregate [sum(Rate#18926) AS total#21345]
+- Filter ((Rate#21349 > cast(1000 as double)) AND (Rate#18926 < cast(100000 as double)))
   +- Join Inner, (Rate#18926 = Rate#21349)
      :- SubqueryAlias df1
      :  +- SubqueryAlias df_table
      :     +- View (`df_table`, [Header_Length#18923,Protocol Type#18924,Duration#18925,Rate#18926,Srate#18927,Drate#18928,fin_flag_number#18929,syn_flag_number#18930,rst_flag_number#18931,psh_flag_number#18932,ack_flag_number#18933,ece_flag_number#18934,cwr_flag_numbe

+--------------------+
|               total|
+--------------------+
|2.238876031062219...|
+--------------------+

== Parsed Logical Plan ==
'Project ['SUM('df1.Rate) AS total#21405]
+- 'Filter (('df2.Rate > 1000) AND ('df1.Rate < 100000))
   +- 'Join Inner, ('df1.Rate = 'df2.Rate)
      :- 'SubqueryAlias df1
      :  +- 'UnresolvedRelation [df_table], [], false
      +- 'SubqueryAlias df2
         +- 'UnresolvedRelation [df_table], [], false

== Analyzed Logical Plan ==
total: double
Aggregate [sum(Rate#18926) AS total#21405]
+- Filter ((Rate#21409 > cast(1000 as double)) AND (Rate#18926 < cast(100000 as double)))
   +- Join Inner, (Rate#18926 = Rate#21409)
      :- SubqueryAlias df1
      :  +- SubqueryAlias df_table
      :     +- View (`df_table`, [Header_Length#18923,Protocol Type#18924,Duration#18925,Rate#18926,Srate#18927,Drate#18928,fin_flag_number#18929,syn_flag_number#18930,rst_flag_number#18931,psh_flag_number#18932,ack_flag_number#18933,ece_flag_number#18934,cwr_flag_numbe

+--------------------+
|               total|
+--------------------+
|2.238876031062219...|
+--------------------+

5.76 s ± 561 ms per loop (mean ± std. dev. of 5 runs, 1 loop each)


In [129]:
df.show(10)

+-------------+-------------+--------+---------+---------+-----+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------+---------+---------+-----------+----+-----+---+------+----+---+---+-----------+---------+----+---+----+----+---+---+-------+----+-------+--------+---------+--------+------------+------+----------+----------+----------+----------+------+--------+
|Header_Length|Protocol Type|Duration|     Rate|    Srate|Drate|fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|ack_count|syn_count|fin_count|  rst_count|HTTP|HTTPS|DNS|Telnet|SMTP|SSH|IRC|        TCP|      UDP|DHCP|ARP|ICMP|IGMP|IPv|LLC|Tot sum| Min|    Max|     AVG|      Std|Tot size|         IAT|Number|  Magnitue|    Radius|Covariance|  Variance|Weight|   label|
+-------------+-------------+--------+---------+---------+-----+---------------+---------------+---------------+---------------+--------

# проверка dataframe vs sql на join

In [136]:
%%timeit -r 5

joined = df.alias("df1").filter(col("Header_Length") > 55).join(df.alias("df2").filter(col("Header_Length") > 55), "Header_Length") \
           .filter(col("df2.Rate") > 1000) \
           .filter(col("df1.Rate") < 100000) \
           .agg(sum("df1.Rate").alias("total"))

joined.show(1000)
joined.explain(True)

+--------------------+
|               total|
+--------------------+
|9.676898815464082E12|
+--------------------+

== Parsed Logical Plan ==
'Aggregate [sum('df1.Rate) AS total#25864]
+- Filter (Rate#18926 < cast(100000 as double))
   +- Filter (Rate#25728 > cast(1000 as double))
      +- Project [Header_Length#18923, Protocol Type#18924, Duration#18925, Rate#18926, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#18935, ack_count#18936, syn_count#18937, fin_count#18938, rst_count#18939, HTTP#18940, HTTPS#18941, DNS#18942, Telnet#18943, SMTP#18944, SSH#18945, IRC#18946, ... 67 more fields]
         +- Join Inner, (Header_Length#18923 = Header_Length#25725)
            :- Filter (Header_Length#18923 > cast(55 as double))
            :  +- SubqueryAlias df1
            :     +- Relation [Header_Length#18923,Protocol Type#18924,Duration#18925,Rate#18926,Srate

+--------------------+
|               total|
+--------------------+
|9.676898815465047E12|
+--------------------+

== Parsed Logical Plan ==
'Aggregate [sum('df1.Rate) AS total#26110]
+- Filter (Rate#18926 < cast(100000 as double))
   +- Filter (Rate#25974 > cast(1000 as double))
      +- Project [Header_Length#18923, Protocol Type#18924, Duration#18925, Rate#18926, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#18935, ack_count#18936, syn_count#18937, fin_count#18938, rst_count#18939, HTTP#18940, HTTPS#18941, DNS#18942, Telnet#18943, SMTP#18944, SSH#18945, IRC#18946, ... 67 more fields]
         +- Join Inner, (Header_Length#18923 = Header_Length#25971)
            :- Filter (Header_Length#18923 > cast(55 as double))
            :  +- SubqueryAlias df1
            :     +- Relation [Header_Length#18923,Protocol Type#18924,Duration#18925,Rate#18926,Srate

+--------------------+
|               total|
+--------------------+
|9.676898815464455E12|
+--------------------+

== Parsed Logical Plan ==
'Aggregate [sum('df1.Rate) AS total#26356]
+- Filter (Rate#18926 < cast(100000 as double))
   +- Filter (Rate#26220 > cast(1000 as double))
      +- Project [Header_Length#18923, Protocol Type#18924, Duration#18925, Rate#18926, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#18935, ack_count#18936, syn_count#18937, fin_count#18938, rst_count#18939, HTTP#18940, HTTPS#18941, DNS#18942, Telnet#18943, SMTP#18944, SSH#18945, IRC#18946, ... 67 more fields]
         +- Join Inner, (Header_Length#18923 = Header_Length#26217)
            :- Filter (Header_Length#18923 > cast(55 as double))
            :  +- SubqueryAlias df1
            :     +- Relation [Header_Length#18923,Protocol Type#18924,Duration#18925,Rate#18926,Srate

+--------------------+
|               total|
+--------------------+
|9.676898815462094E12|
+--------------------+

== Parsed Logical Plan ==
'Aggregate [sum('df1.Rate) AS total#26602]
+- Filter (Rate#18926 < cast(100000 as double))
   +- Filter (Rate#26466 > cast(1000 as double))
      +- Project [Header_Length#18923, Protocol Type#18924, Duration#18925, Rate#18926, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#18935, ack_count#18936, syn_count#18937, fin_count#18938, rst_count#18939, HTTP#18940, HTTPS#18941, DNS#18942, Telnet#18943, SMTP#18944, SSH#18945, IRC#18946, ... 67 more fields]
         +- Join Inner, (Header_Length#18923 = Header_Length#26463)
            :- Filter (Header_Length#18923 > cast(55 as double))
            :  +- SubqueryAlias df1
            :     +- Relation [Header_Length#18923,Protocol Type#18924,Duration#18925,Rate#18926,Srate

+--------------------+
|               total|
+--------------------+
|9.676898815465916E12|
+--------------------+

== Parsed Logical Plan ==
'Aggregate [sum('df1.Rate) AS total#26848]
+- Filter (Rate#18926 < cast(100000 as double))
   +- Filter (Rate#26712 > cast(1000 as double))
      +- Project [Header_Length#18923, Protocol Type#18924, Duration#18925, Rate#18926, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#18935, ack_count#18936, syn_count#18937, fin_count#18938, rst_count#18939, HTTP#18940, HTTPS#18941, DNS#18942, Telnet#18943, SMTP#18944, SSH#18945, IRC#18946, ... 67 more fields]
         +- Join Inner, (Header_Length#18923 = Header_Length#26709)
            :- Filter (Header_Length#18923 > cast(55 as double))
            :  +- SubqueryAlias df1
            :     +- Relation [Header_Length#18923,Protocol Type#18924,Duration#18925,Rate#18926,Srate

+--------------------+
|               total|
+--------------------+
|9.676898815460053E12|
+--------------------+

== Parsed Logical Plan ==
'Aggregate [sum('df1.Rate) AS total#27094]
+- Filter (Rate#18926 < cast(100000 as double))
   +- Filter (Rate#26958 > cast(1000 as double))
      +- Project [Header_Length#18923, Protocol Type#18924, Duration#18925, Rate#18926, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece_flag_number#18934, cwr_flag_number#18935, ack_count#18936, syn_count#18937, fin_count#18938, rst_count#18939, HTTP#18940, HTTPS#18941, DNS#18942, Telnet#18943, SMTP#18944, SSH#18945, IRC#18946, ... 67 more fields]
         +- Join Inner, (Header_Length#18923 = Header_Length#26955)
            :- Filter (Header_Length#18923 > cast(55 as double))
            :  +- SubqueryAlias df1
            :     +- Relation [Header_Length#18923,Protocol Type#18924,Duration#18925,Rate#18926,Srate

In [ ]:
%%timeit -r 5

df.createOrReplaceTempView("df_table")

sql_result = spark.sql("""
    WITH 
  filtered_df1 AS (
      SELECT * FROM df_table 
      WHERE Header_Length > 55 AND Rate < 100000
  ),
  filtered_df2 AS (
      SELECT * FROM df_table 
      WHERE Header_Length > 55 AND Rate > 1000
  )
  SELECT SUM(filtered_df1.Rate) as total
  FROM filtered_df1
  JOIN filtered_df2 ON filtered_df1.Header_Length = filtered_df2.Header_Length
""")

sql_result.explain(True)

sql_result.show(1000)

== Parsed Logical Plan ==
CTE [filtered_df1, filtered_df2]
:  :- 'SubqueryAlias filtered_df1
:  :  +- 'Project [*]
:  :     +- 'Filter (('Header_Length > 55) AND ('Rate < 100000))
:  :        +- 'UnresolvedRelation [df_table], [], false
:  +- 'SubqueryAlias filtered_df2
:     +- 'Project [*]
:        +- 'Filter (('Header_Length > 55) AND ('Rate > 1000))
:           +- 'UnresolvedRelation [df_table], [], false
+- 'Project ['SUM('filtered_df1.Rate) AS total#27201]
   +- 'Join Inner, ('filtered_df1.Header_Length = 'filtered_df2.Header_Length)
      :- 'UnresolvedRelation [filtered_df1], [], false
      +- 'UnresolvedRelation [filtered_df2], [], false

== Analyzed Logical Plan ==
total: double
WithCTE
:- CTERelationDef 18, false
:  +- SubqueryAlias filtered_df1
:     +- Project [Header_Length#18923, Protocol Type#18924, Duration#18925, Rate#18926, Srate#18927, Drate#18928, fin_flag_number#18929, syn_flag_number#18930, rst_flag_number#18931, psh_flag_number#18932, ack_flag_number#18933, ece

+-------------------+
|              total|
+-------------------+
|9.67689881546491E12|
+-------------------+

== Parsed Logical Plan ==
CTE [filtered_df1, filtered_df2]
:  :- 'SubqueryAlias filtered_df1
:  :  +- 'Project [*]
:  :     +- 'Filter (('Header_Length > 55) AND ('Rate < 100000))
:  :        +- 'UnresolvedRelation [df_table], [], false
:  +- 'SubqueryAlias filtered_df2
:     +- 'Project [*]
:        +- 'Filter (('Header_Length > 55) AND ('Rate > 1000))
:           +- 'UnresolvedRelation [df_table], [], false
+- 'Project ['SUM('filtered_df1.Rate) AS total#27264]
   +- 'Join Inner, ('filtered_df1.Header_Length = 'filtered_df2.Header_Length)
      :- 'UnresolvedRelation [filtered_df1], [], false
      +- 'UnresolvedRelation [filtered_df2], [], false

== Analyzed Logical Plan ==
total: double
WithCTE
:- CTERelationDef 20, false
:  +- SubqueryAlias filtered_df1
:     +- Project [Header_Length#18923, Protocol Type#18924, Duration#18925, Rate#18926, Srate#18927, Drate#18928, fin_fla

+-------------------+
|              total|
+-------------------+
|9.67689881546304E12|
+-------------------+

== Parsed Logical Plan ==
CTE [filtered_df1, filtered_df2]
:  :- 'SubqueryAlias filtered_df1
:  :  +- 'Project [*]
:  :     +- 'Filter (('Header_Length > 55) AND ('Rate < 100000))
:  :        +- 'UnresolvedRelation [df_table], [], false
:  +- 'SubqueryAlias filtered_df2
:     +- 'Project [*]
:        +- 'Filter (('Header_Length > 55) AND ('Rate > 1000))
:           +- 'UnresolvedRelation [df_table], [], false
+- 'Project ['SUM('filtered_df1.Rate) AS total#27327]
   +- 'Join Inner, ('filtered_df1.Header_Length = 'filtered_df2.Header_Length)
      :- 'UnresolvedRelation [filtered_df1], [], false
      +- 'UnresolvedRelation [filtered_df2], [], false

== Analyzed Logical Plan ==
total: double
WithCTE
:- CTERelationDef 22, false
:  +- SubqueryAlias filtered_df1
:     +- Project [Header_Length#18923, Protocol Type#18924, Duration#18925, Rate#18926, Srate#18927, Drate#18928, fin_fla

+--------------------+
|               total|
+--------------------+
|9.676898815462936E12|
+--------------------+

== Parsed Logical Plan ==
CTE [filtered_df1, filtered_df2]
:  :- 'SubqueryAlias filtered_df1
:  :  +- 'Project [*]
:  :     +- 'Filter (('Header_Length > 55) AND ('Rate < 100000))
:  :        +- 'UnresolvedRelation [df_table], [], false
:  +- 'SubqueryAlias filtered_df2
:     +- 'Project [*]
:        +- 'Filter (('Header_Length > 55) AND ('Rate > 1000))
:           +- 'UnresolvedRelation [df_table], [], false
+- 'Project ['SUM('filtered_df1.Rate) AS total#27390]
   +- 'Join Inner, ('filtered_df1.Header_Length = 'filtered_df2.Header_Length)
      :- 'UnresolvedRelation [filtered_df1], [], false
      +- 'UnresolvedRelation [filtered_df2], [], false

== Analyzed Logical Plan ==
total: double
WithCTE
:- CTERelationDef 24, false
:  +- SubqueryAlias filtered_df1
:     +- Project [Header_Length#18923, Protocol Type#18924, Duration#18925, Rate#18926, Srate#18927, Drate#18928, fi

+--------------------+
|               total|
+--------------------+
|9.676898815462455E12|
+--------------------+

== Parsed Logical Plan ==
CTE [filtered_df1, filtered_df2]
:  :- 'SubqueryAlias filtered_df1
:  :  +- 'Project [*]
:  :     +- 'Filter (('Header_Length > 55) AND ('Rate < 100000))
:  :        +- 'UnresolvedRelation [df_table], [], false
:  +- 'SubqueryAlias filtered_df2
:     +- 'Project [*]
:        +- 'Filter (('Header_Length > 55) AND ('Rate > 1000))
:           +- 'UnresolvedRelation [df_table], [], false
+- 'Project ['SUM('filtered_df1.Rate) AS total#27453]
   +- 'Join Inner, ('filtered_df1.Header_Length = 'filtered_df2.Header_Length)
      :- 'UnresolvedRelation [filtered_df1], [], false
      +- 'UnresolvedRelation [filtered_df2], [], false

== Analyzed Logical Plan ==
total: double
WithCTE
:- CTERelationDef 26, false
:  +- SubqueryAlias filtered_df1
:     +- Project [Header_Length#18923, Protocol Type#18924, Duration#18925, Rate#18926, Srate#18927, Drate#18928, fi

+--------------------+
|               total|
+--------------------+
|9.676898815463246E12|
+--------------------+

== Parsed Logical Plan ==
CTE [filtered_df1, filtered_df2]
:  :- 'SubqueryAlias filtered_df1
:  :  +- 'Project [*]
:  :     +- 'Filter (('Header_Length > 55) AND ('Rate < 100000))
:  :        +- 'UnresolvedRelation [df_table], [], false
:  +- 'SubqueryAlias filtered_df2
:     +- 'Project [*]
:        +- 'Filter (('Header_Length > 55) AND ('Rate > 1000))
:           +- 'UnresolvedRelation [df_table], [], false
+- 'Project ['SUM('filtered_df1.Rate) AS total#27516]
   +- 'Join Inner, ('filtered_df1.Header_Length = 'filtered_df2.Header_Length)
      :- 'UnresolvedRelation [filtered_df1], [], false
      +- 'UnresolvedRelation [filtered_df2], [], false

== Analyzed Logical Plan ==
total: double
WithCTE
:- CTERelationDef 28, false
:  +- SubqueryAlias filtered_df1
:     +- Project [Header_Length#18923, Protocol Type#18924, Duration#18925, Rate#18926, Srate#18927, Drate#18928, fi

+--------------------+
|               total|
+--------------------+
|9.676898815462477E12|
+--------------------+

10.5 s ± 314 ms per loop (mean ± std. dev. of 5 runs, 1 loop each)
